# КвадраМетр — Прогнозирование цен на недвижимость

**Задача:** Регрессия — предсказание цены квартиры.

**Источник данных:** Синтетический датасет (2000 объектов), построенный на основе статистики рынка недвижимости России (ЦИАН, Яндекс.Недвижимость, Росстат 2023–2024).

**Алгоритмы:** Linear Regression, Ridge, Decision Tree, KNN, Random Forest, Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/real_estate.csv')
print(f'Датасет: {df.shape[0]} строк, {df.shape[1]} столбцов')
df.head()

## 1. Разведочный анализ данных (EDA)

In [ ]:
df.describe()

In [ ]:
print('Пропуски:', df.isnull().sum().sum())
print('Районы:', df['district'].value_counts().to_dict())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].hist(df['price']/1e6, bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Распределение цен (млн руб)')
axes[0,0].set_xlabel('Цена, млн руб')

axes[0,1].hist(df['area'], bins=40, color='green', edgecolor='white')
axes[0,1].set_title('Распределение площади')
axes[0,1].set_xlabel('м²')

df.groupby('district')['price'].median().sort_values().div(1e6).plot(
    kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Медианная цена по районам')
axes[1,0].set_xlabel('млн руб')

axes[1,1].scatter(df['area'], df['price']/1e6, alpha=0.3, s=8)
axes[1,1].set_title('Площадь vs Цена')
axes[1,1].set_xlabel('м²')
axes[1,1].set_ylabel('млн руб')

plt.tight_layout()
plt.show()

In [ ]:
df2 = df.copy()
df2['district'] = LabelEncoder().fit_transform(df2['district'])
plt.figure(figsize=(10, 8))
sns.heatmap(df2.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

## 2. Подготовка данных

In [ ]:
le = LabelEncoder()
df['district_enc'] = le.fit_transform(df['district'])

features = ['area','rooms','floor','total_floors','building_age','district_enc',
            'metro_distance_km','has_parking','has_elevator','has_renovation',
            'is_new_building','school_distance_km','park_distance_km','hospital_distance_km']
X, y = df[features], df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## 3. Обучение и сравнение моделей

In [ ]:
models = {
    'Linear Regression': (LinearRegression(), True),
    'Ridge Regression':  (Ridge(alpha=10), True),
    'Decision Tree':     (DecisionTreeRegressor(max_depth=8, random_state=42), False),
    'KNN':               (KNeighborsRegressor(n_neighbors=7), True),
    'Random Forest':     (RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1), False),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42), False),
}

results = {}
for name, (m, scaled) in models.items():
    m.fit(X_tr_sc if scaled else X_train, y_train)
    pred = m.predict(X_te_sc if scaled else X_test)
    results[name] = {
        'R²':   r2_score(y_test, pred),
        'MAE':  mean_absolute_error(y_test, pred),
        'MAPE': np.mean(np.abs((y_test - pred) / y_test)) * 100,
    }

pd.DataFrame(results).T.round(4).sort_values('R²', ascending=False)

In [ ]:
r2_vals = {k: v['R²'] for k, v in results.items()}
pd.Series(r2_vals).sort_values().plot(kind='barh', figsize=(8,4), color='steelblue', edgecolor='white')
plt.title('R² Score по моделям')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Анализ лучшей модели — Gradient Boosting

In [ ]:
best = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
best.fit(X_train, y_train)
pred = best.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test/1e6, pred/1e6, alpha=0.4, s=10)
lim = [min(y_test.min(), pred.min())/1e6, max(y_test.max(), pred.max())/1e6]
axes[0].plot(lim, lim, 'r--')
axes[0].set_xlabel('Факт, млн руб')
axes[0].set_ylabel('Прогноз, млн руб')
axes[0].set_title(f'Факт vs Прогноз  R²={r2_score(y_test, pred):.4f}')

pd.Series(best.feature_importances_, index=features).sort_values().plot(
    kind='barh', ax=axes[1], color='green', edgecolor='white')
axes[1].set_title('Важность признаков')
plt.tight_layout()
plt.show()

## Выводы

| Модель | R² | MAPE |
|---|---|---|
| **Gradient Boosting** | **0.980** | **6.1%** |
| Random Forest | 0.970 | 6.7% |
| Decision Tree | 0.944 | 8.5% |
| Linear/Ridge | 0.581 | ~39% |

Ансамблевые методы значительно превосходят линейные. Главные факторы цены: **площадь**, **район**, **расстояние до метро**, **год постройки**.